# Modern LangChain + Groq Workshop: Modules 8–15
**Project:** Build a technical support assistant grounded in CIS Windows security benchmark documentation.

**LLM:** Groq via `ChatGroq` · **Composition:** modern LangChain and LCEL · **Embeddings:** Hugging Face locally or OpenAI API · **Vector store:** `InMemoryVectorStore`

## Learning outcomes
- Batch independent model calls.
- Load and split PDF, text, and web documents.
- Generate embeddings and index chunks.
- Retrieve relevant evidence from a vector store.
- Build a basic RAG pipeline with LCEL.
- Compare baseline retrieval with query expansion, reranking, and hybrid retrieval.

| Module | Topic | Practical |
|---:|---|---|
| 8 | Batch processing | Process multiple learner/support questions |
| 9 | Document loading | Load PDF, text, and web documents |
| 10 | Text splitting | Chunk security guidance |
| 11 | Embeddings | Represent chunks and queries as vectors |
| 12 | Vector stores | Index and search chunks |
| 13 | Retrievers | Retrieve evidence |
| 14 | Basic RAG | Answer from retrieved context using Groq |
| 15 | RAG improvements | Compare query expansion, reranking, hybrid retrieval |

**Notes:** Every chapter includes imports, a demonstration, and student hands-on work. Code cells make live API calls when run. Use only documents you are licensed to process. Synthetic text is included for a reliable first run.

## Setup — install dependencies
Run once. Restart the kernel if prompted.

In [ ]:
%pip install -U langchain langchain-core langchain-groq langchain-community langchain-text-splitters langchain-huggingface langchain-openai pypdf beautifulsoup4 sentence-transformers rank-bm25

## Configure Groq
Set `GROQ_API_KEY` in your environment or enter it when prompted. `GROQ_MODEL` can override the default model ID if needed. Never commit keys to source control.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter GROQ_API_KEY: ")

GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")

from langchain_groq import ChatGroq
llm = ChatGroq(model=GROQ_MODEL, temperature=0.1)
print(f"Configured ChatGroq: {GROQ_MODEL}")

# Module 8 — Batch Processing

## Notes
**Scenario:** A training team needs short responses to many independent learner questions.

- `batch()` and `abatch()` execute a Runnable against multiple inputs.
- Each input is processed independently; this is not one joint conversation.
- LCEL makes the prompt → model → parser pipeline reusable.
- In production, account for rate limits, concurrency, retries, and partial failures.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

support_prompt = ChatPromptTemplate.from_template(
    "You are a technical training support assistant. Answer in at most 70 words.\nQuestion: {question}"
)
support_chain = support_prompt | llm | StrOutputParser()

questions = [
    {"question": "I know Python and SQL. What should I learn next for data engineering?"},
    {"question": "What is the difference between RAG and fine-tuning?"},
    {"question": "Why do LLM applications use embeddings?"},
]
answers = support_chain.batch(questions)
for i, (item, answer) in enumerate(zip(questions, answers), 1):
    print(f"Q{i}: {item['question']}\nA: {answer}\n")

### Demonstration — asynchronous batch
Jupyter supports top-level `await`.

In [ ]:
async_answers = await support_chain.abatch(questions)
for i, answer in enumerate(async_answers, 1):
    print(f"Async answer {i}: {answer}\n")

### Hands-on — Module 8
1. Add three questions about Windows hardening or RAG.
2. Ask for a short answer plus one suggested next step.
3. Compare sequential calls with `batch()`.
4. Discuss how you would handle rate limits and partial failures for hundreds of inputs.

# Module 9 — Document Loading

## Notes
**Scenario:** Build a knowledge base from a licensed CIS benchmark PDF, supplementary text files, and approved web pages.

- `PyPDFLoader` loads PDF pages, often with page metadata.
- `TextLoader` loads plain text.
- `WebBaseLoader` retrieves and parses web pages; network access and compatible page HTML are required.
- Loaders return `Document` objects with `page_content` and `metadata`.
- Respect document licensing, website terms, and access policies.
- The synthetic text below makes the first run independent of external files.

In [ ]:
import os
from pathlib import Path
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader, PyPDFLoader, WebBaseLoader

sample_text = '''
Synthetic Windows security training notes — not official CIS guidance.
Use unique user accounts and apply least privilege.
Enable security logging appropriate to organisational policy.
Apply security updates through a tested patch-management process.
Review firewall rules and disable services that are not required.
Validate configuration changes in a test environment before deployment.
'''
sample_path = Path("sample_windows_security_notes.txt")
sample_path.write_text(sample_text, encoding="utf-8")
text_docs = TextLoader(str(sample_path), encoding="utf-8").load()
print("Loaded text documents:", len(text_docs))
print("Metadata:", text_docs[0].metadata)
print(text_docs[0].page_content)

### Demonstration — optional PDF and web loading
Set `PDF_PATH` or `RAG_DEMO_WEB_URL` before running.

In [ ]:
PDF_PATH = os.getenv("PDF_PATH", "")
if PDF_PATH and Path(PDF_PATH).exists():
    pdf_docs = PyPDFLoader(PDF_PATH).load()
    print("PDF pages:", len(pdf_docs))
    print("First page metadata:", pdf_docs[0].metadata if pdf_docs else {})
else:
    print("PDF skipped: set PDF_PATH to a licensed local PDF.")

WEB_URL = os.getenv("RAG_DEMO_WEB_URL", "")
if WEB_URL:
    web_docs = WebBaseLoader(WEB_URL).load()
    print("Web documents:", len(web_docs))
    print("First web metadata:", web_docs[0].metadata if web_docs else {})
else:
    print("Web skipped: set RAG_DEMO_WEB_URL to an approved page.")

### Hands-on — Module 9
1. Create `team_security_policy.txt` with 5–10 policy statements and load it using `TextLoader`.
2. If you have a licensed PDF, load it with `PyPDFLoader`.
3. Try an approved public web page with `WebBaseLoader`.
4. Compare document count, text length, and metadata across sources.

# Module 10 — Text Splitting

## Notes
**Scenario:** Long security documents need to be divided into chunks suitable for embedding and retrieval.

- `RecursiveCharacterTextSplitter` tries a hierarchy of separators.
- `chunk_size` sets approximate chunk length; `chunk_overlap` repeats text between adjacent chunks.
- `split_documents()` preserves document metadata on chunks.
- Evaluate chunk sizes: small chunks may lose context; large chunks may dilute retrieval precision.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=220,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(text_docs)
print("Source documents:", len(text_docs), "| chunks:", len(chunks))
for i, chunk in enumerate(chunks, 1):
    print(f"\nCHUNK {i} | metadata={chunk.metadata}\n{chunk.page_content}")

### Hands-on — Module 10
1. Try `chunk_size=100`, `chunk_overlap=10`.
2. Try `chunk_size=400`, `chunk_overlap=60`.
3. Compare chunk count and context preservation.
4. Write down a chunking strategy you would test on a real security benchmark.

# Module 11 — Embeddings

## Notes
**Scenario:** Represent document chunks and user questions as vectors for semantic matching.

- Embedding models are distinct from chat LLMs. Groq chat access does not imply Groq embedding access.
- `HuggingFaceEmbeddings` runs locally after downloading a model; the first run may take time.
- `OpenAIEmbeddings` calls a hosted embedding API and requires `OPENAI_API_KEY`.
- `embed_documents()` embeds many texts; `embed_query()` embeds one query.
- Use the same embedding model for indexing and querying.

In [ ]:
import os
from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_MODEL_NAME = os.getenv(
    "EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2"
)
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)

texts = [chunk.page_content for chunk in chunks]
document_vectors = embeddings.embed_documents(texts)
query_vector = embeddings.embed_query("How should security updates be managed?")
print("Embedded chunks:", len(document_vectors))
print("Document vector dimensions:", len(document_vectors[0]) if document_vectors else 0)
print("Query vector dimensions:", len(query_vector))

### Optional alternative — OpenAI hosted embeddings
Use only if you have an OpenAI API key. This is separate from the Groq key.

In [ ]:
# Uncomment to use OpenAI embeddings instead:
# from langchain_openai import OpenAIEmbeddings
# if not os.environ.get("OPENAI_API_KEY"):
#     os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ")
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# document_vectors = embeddings.embed_documents(texts)
# query_vector = embeddings.embed_query("How should security updates be managed?")

### Hands-on — Module 11
1. Embed at least five policy statements.
2. Print the vector dimension and compare with another model if available.
3. Explain why indexing and query-time embedding must use the same model.
4. Discuss privacy and operational trade-offs between local and hosted embeddings.

# Module 12 — Vector Stores

## Notes
**Scenario:** Index chunks and search for passages relevant to a security question.

- `InMemoryVectorStore` is convenient for learning but is not durable across process restarts.
- `add_documents()` indexes documents with the configured embedding model.
- `similarity_search()` retrieves semantically similar documents; `k` controls the number returned.
- Inspect the retrieved text and metadata, not just the result count.

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(chunks)

query = "How should the organisation handle security updates?"
matches = vector_store.similarity_search(query, k=3)
for rank, doc in enumerate(matches, 1):
    print(f"\nMATCH {rank} | metadata={doc.metadata}\n{doc.page_content}")

### Hands-on — Module 12
1. Add two new synthetic policy documents to the vector store.
2. Search using exact wording and then a paraphrase.
3. Compare `k=1`, `k=3`, and `k=5`.
4. Explain why an in-memory store may not fit production needs.

# Module 13 — Retrievers

## Notes
**Scenario:** Wrap vector search in a reusable interface so downstream RAG code need not manage search details directly.

- `vector_store.as_retriever()` creates a retriever.
- `retriever.invoke(question)` returns relevant `Document` objects.
- `search_kwargs={"k": ...}` configures the number of documents retrieved.
- Retrieval quality depends on chunking, embeddings, search configuration, and query wording.

In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)
retrieved_docs = retriever.invoke("Which practices help keep Windows systems secure?")
print("Retrieved:", len(retrieved_docs))
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\nDOCUMENT {i} | metadata={doc.metadata}\n{doc.page_content}")

### Hands-on — Module 13
1. Compare retrievers with `k=1`, `k=3`, and `k=5`.
2. Try another supported search configuration.
3. Record whether extra documents add useful evidence or irrelevant context.

# Module 14 — Basic RAG with LCEL

## Notes
**Scenario:** Answer a security question using retrieved policy text rather than relying only on pretrained model knowledge.

Pipeline: retrieve documents → format context → prompt → Groq model → parse answer.

This chapter uses manual LCEL composition to make each step explicit. Older helpers such as `create_retrieval_chain()` and `create_stuff_documents_chain()` are useful to recognise in existing code, but manual Runnable composition is the primary example here.

**Safety note:** Ask the model to acknowledge when context is insufficient. This reduces risk but does not guarantee that hallucinations are eliminated.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(
        f"Source metadata: {doc.metadata}\nContent: {doc.page_content}"
        for doc in docs
    )

rag_prompt = ChatPromptTemplate.from_template(
    "Answer using only the context below. If evidence is insufficient, say so. "
    "Do not invent security requirements.\n\nContext:\n{context}\n\nQuestion: {question}"
)

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

rag_question = "How should security updates be managed?"
print(rag_chain.invoke(rag_question))

### Demonstration — inspect retrieved context
Separate retrieval from generation to diagnose failures.

In [ ]:
context_docs = retriever.invoke(rag_question)
print(format_docs(context_docs))

### Hands-on — Module 14
1. Ask three questions answerable from the sample documents.
2. Ask one question not covered by the documents.
3. Record whether the model acknowledges missing evidence.
4. Modify the prompt to request bullet points and source metadata.
5. Explain the difference between retrieval failure and generation failure.

# Module 15 — RAG Improvements

## Notes
**Scenario:** The baseline RAG assistant misses relevant passages when the question uses different wording or several passages compete for relevance.

- **Multi-query / query expansion:** generate alternative search queries to improve recall.
- **Reranking:** reorder candidates using query-document relevance.
- **Hybrid retrieval:** combine lexical search (e.g., BM25) with vector similarity.
- **Evaluation:** compare approaches on the same questions and inspect whether relevant evidence appears in the top results.

These methods add latency, dependencies, and sometimes cost. Measure before adopting them. Integration import paths can vary with LangChain versions.

In [ ]:
import re

def show_retrieval(label, docs):
    print(f"\n=== {label} ({len(docs)} docs) ===")
    for i, doc in enumerate(docs, 1):
        print(f"{i}. {doc.page_content[:240]!r}")
        print("   metadata:", doc.metadata)

test_query = "What should we do to keep operating systems patched?"
baseline_docs = retriever.invoke(test_query)
show_retrieval("Baseline vector retrieval", baseline_docs)

### Optional demonstration — MultiQueryRetriever
Check the installed version's docs if the import path differs.

In [ ]:
# Optional: query expansion using an LLM.
# Some current/legacy installations expose this through langchain_classic.
#
# from langchain_classic.retrievers import MultiQueryRetriever
# multi_query_retriever = MultiQueryRetriever.from_llm(
#     retriever=vector_store.as_retriever(search_kwargs={"k": 3}),
#     llm=llm,
# )
# expanded_docs = multi_query_retriever.invoke(test_query)
# show_retrieval("Multi-query retrieval", expanded_docs)

### Demonstration — keyword-overlap baseline

This is an illustration of lexical matching, **not** BM25 or a production hybrid retriever. It shows how keyword matching can rank passages differently from semantic search.

In [ ]:
def lexical_score(query: str, document: str) -> int:
    query_terms = set(re.findall(r"[a-zA-Z0-9]+", query.lower()))
    doc_terms = set(re.findall(r"[a-zA-Z0-9]+", document.lower()))
    return len(query_terms & doc_terms)

lexical_ranked = sorted(
    chunks,
    key=lambda doc: lexical_score(test_query, doc.page_content),
    reverse=True,
)
show_retrieval("Simple keyword-overlap ranking", lexical_ranked[:3])

### Hands-on — Module 15
1. Write five test questions: two direct matches, two paraphrases, and one unsupported question.
2. Compare vector retrieval against multi-query retrieval if available.
3. Check whether relevant evidence appears in the top three results.
4. Propose a reranking stage for the top ten candidates.
5. Define how you would evaluate recall@k, faithfulness, latency, and API cost.

**Important:** For a real hybrid retriever, use a supported lexical retriever such as BM25 and combine its results with vector retrieval using a documented ensemble/fusion approach.

# Final Workshop Review

## End-to-end pipeline
`Documents → Chunks → Embeddings → Vector store → Retriever → LCEL RAG chain → Groq answer`

## Troubleshooting
- Authentication: check `GROQ_API_KEY` and any embedding provider key.
- Model errors: check `GROQ_MODEL` is available to your account.
- PDF errors: verify file path, readability, and permissions.
- Web errors: check network access and site retrieval policies.
- Embedding errors: check model download and local dependencies.
- Poor retrieval: inspect chunks, metadata, embedding model, query, and top-k.
- Weak answer: inspect retrieved context before changing the prompt.
- Version mismatch: check installed versions and current documentation.

## Next steps
Add a persistent vector database, metadata filters, document versioning, RAG evaluation, proper hybrid retrieval, reranking, and eventually LangGraph workflows and agents.